# Hybrid Global + Local Attention — Solution

Reference implementation for `02_hybrid_global_local_exercise.ipynb`.

## Setup

In [7]:
import torch
import torch.nn as nn

## Solution 1 — `HybridAttentionStack`

In [8]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, causal=False):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.causal = causal

        self.qkv_projection = nn.Linear(d_model, d_model * 3)
        self.out_projection = nn.Linear(d_model, d_model)

    def forward(self, x):
        bs, seq_len, _ = x.shape

        Q, K, V = torch.chunk(self.qkv_projection(x), 3, dim=-1)
        Q = Q.view(bs, seq_len, self.n_heads, self.d_head).permute(0, 2, 1, 3)
        K = K.view(bs, seq_len, self.n_heads, self.d_head).permute(0, 2, 1, 3)
        V = V.view(bs, seq_len, self.n_heads, self.d_head).permute(0, 2, 1, 3)

        scores = Q @ K.transpose(-2, -1) / self.d_head**0.5

        if self.causal:
            mask = torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool))
            scores = scores.masked_fill(~mask, float('-inf'))

        weights = torch.softmax(scores, dim=-1)
        out = weights @ V
        out = out.permute(0, 2, 1, 3).contiguous().view(bs, seq_len, self.d_model)
        return self.out_projection(out)

In [9]:
class SlidingWindowAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_len, window_size):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.window_size = window_size
        self.d_head = d_model // n_heads

        self.qkv_proj = nn.Linear(d_model, d_model * 3)
        self.out_proj = nn.Linear(d_model, d_model)

        mask = torch.arange(max_len)[:, None] - torch.arange(max_len)[None, :]
        self.register_buffer(
            'sliding_window_mask',
            torch.where((mask >= window_size) | (mask < 0), True, False),
        )

    def forward(self, x):
        bs, seq_len, _ = x.shape

        Q, K, V = torch.chunk(self.qkv_proj(x), 3, dim=-1)
        Q = Q.view(bs, seq_len, self.n_heads, self.d_head).permute(0, 2, 1, 3)
        K = K.view(bs, seq_len, self.n_heads, self.d_head).permute(0, 2, 1, 3)
        V = V.view(bs, seq_len, self.n_heads, self.d_head).permute(0, 2, 1, 3)

        scores = Q @ K.transpose(-2, -1) / self.d_head**0.5
        scores = scores.masked_fill(
            self.sliding_window_mask[:seq_len, :seq_len], float('-inf')
        )

        weights = torch.softmax(scores, dim=-1)
        out = weights @ V
        out = out.permute(0, 2, 1, 3).contiguous().view(bs, seq_len, self.d_model)
        return self.out_proj(out)

In [10]:
class HybridAttentionStack(nn.Module):
    def __init__(self, d_model, n_heads, n_layers, max_len, window_size, global_every=6):
        super().__init__()
        self.attention_layers = nn.ModuleList([
            SlidingWindowAttention(d_model, n_heads, max_len, window_size)
            if i % global_every != 0
            else MultiHeadAttention(d_model, n_heads, causal=True)
            for i in range(n_layers)
        ])

    def forward(self, x):
        for layer in self.attention_layers:
            x = layer(x)
        return x

**Why 1:5 (or 1:6)?** This ratio preserves ~85% of the cache savings (most layers still bounded), while adding back enough "highway" layers to enable long-range information flow. Tunable: more global = better long-range, more memory; fewer global = cheaper but worse long-range.

**The KV cache implication.** For a 26-layer Gemma 3 model with 4 global layers: only those 4 layers grow KV cache linearly with sequence length. The other 22 are bounded at `window_size`. So at 128K context, the cache is dominated by 4 layers' worth of full-length cache, not 26.

## Solution 2 — Verify info travels through global layers

**The claim:** an all-local stack has receptive field bounded by `n_layers × (window_size − 1)` (the bucket-brigade reach). A hybrid stack — even with just one global layer — can move information from any token to any later token in a single hop through that global layer.

**The experiment:** build both stacks at the same width/depth. Perturb only one early position. Check whether a position *well beyond* the local receptive field responds.

In [11]:
# Sanity: forward pass produces correct shape
stack = HybridAttentionStack(
    d_model=64, n_heads=8, n_layers=12,
    max_len=64, window_size=4, global_every=6,
)
out = stack(torch.randn(2, 32, 64))
assert out.shape == (2, 32, 64)
print('shape OK:', tuple(out.shape))

shape OK: (2, 32, 64)


In [12]:
torch.manual_seed(0)

d_model    = 64
n_heads    = 8
n_layers   = 4
window     = 4
seq_len    = 32
perturb_at = 0

# Local receptive field after n_layers = n_layers * (window - 1) = 4 * 3 = 12.
# Pick a target position well beyond it so an all-local stack provably cannot reach it.
far_pos = 25
assert far_pos > n_layers * (window - 1), 'far_pos must exceed local receptive field'

# All-local stack: pure SlidingWindowAttention, n_layers deep.
all_local = nn.Sequential(*[
    SlidingWindowAttention(d_model, n_heads, max_len=seq_len, window_size=window)
    for _ in range(n_layers)
])

# Hybrid stack at same depth/width — global_every=2 makes layers 0 and 2 global.
hybrid = HybridAttentionStack(
    d_model=d_model, n_heads=n_heads, n_layers=n_layers,
    max_len=seq_len, window_size=window, global_every=2,
)

x_a = torch.randn(1, seq_len, d_model)
x_b = x_a.clone()
x_b[:, perturb_at, :] = torch.randn(d_model)   # change only one early position

with torch.no_grad():
    out_a_local,  out_b_local  = all_local(x_a),  all_local(x_b)
    out_a_hybrid, out_b_hybrid = hybrid(x_a),     hybrid(x_b)

# All-local: information from pos 0 cannot reach pos 25 (bucket-brigade reach = 12).
assert torch.allclose(out_a_local[0, far_pos], out_b_local[0, far_pos]), \
    'all-local stack should NOT propagate info beyond receptive field'

# Hybrid: even one global layer lets information travel anywhere.
assert not torch.allclose(out_a_hybrid[0, far_pos], out_b_hybrid[0, far_pos]), \
    'hybrid stack SHOULD propagate info via global highway'

delta_local  = (out_a_local[0, far_pos]  - out_b_local[0, far_pos]).abs().max().item()
delta_hybrid = (out_a_hybrid[0, far_pos] - out_b_hybrid[0, far_pos]).abs().max().item()
print(f'all-local  out[0, {far_pos}] max delta: {delta_local:.2e}   (should be ~0)')
print(f'hybrid     out[0, {far_pos}] max delta: {delta_hybrid:.2e}   (should be >> 0)')
print('\nAll-local confines info to its receptive field; hybrid does not.')

all-local  out[0, 25] max delta: 0.00e+00   (should be ~0)
hybrid     out[0, 25] max delta: 1.06e-02   (should be >> 0)

All-local confines info to its receptive field; hybrid does not.


## Run the tests

In [13]:
from tests import run_hybrid_tests
run_hybrid_tests(HybridAttentionStack)

Running HybridAttentionStack...
  ✓ Output shape correct
  ✓ Has parameters
  ✓ Gradients flow

  All 3 checks passed ✓



True